# Lab 5b: Workflow Multi-Agente no Foundry (15 min)

## Objetivos
- Criar **agentes no Foundry Agent Service** (server-side, visíveis no portal)
- Usar **Connected Agents** para ligar agentes entre si
- Criar um **orquestrador** que delega tarefas sequencialmente a sub-agentes
- Ver os **run steps** para acompanhar a delegação agente → agente

## Conceitos

### Multi-Agent Workflow no Foundry Agent Service

O Foundry permite criar **workflows multi-agente server-side** onde:
- Cada agente é criado no serviço (visível no portal Foundry)
- Um **orquestrador** usa `ConnectedAgentTool` para delegar tarefas a sub-agentes
- O serviço gere automaticamente a comunicação e contexto entre agentes

### Padrão Sequential Workflow com Connected Agents
```
                         Foundry Agent Service
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   ┌──────────────┐                                              │
│   │ Orquestrador │──── ConnectedAgentTool ──┐                   │
│   └──────┬───────┘                          │                   │
│          │                                  │                   │
│    ┌─────▼──────┐  ┌──────────┐  ┌─────────▼─┐                 │
│    │Investigador│→ │ Arquiteto│→ │  Redator  │                  │
│    └────────────┘  └──────────┘  └───────────┘                  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

O orquestrador decide a **ordem de chamada** com base nas suas instruções.

📖 Docs: https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/workflow

In [4]:
# Setup — mesmo padrão do Lab 3.1
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.agents.models import ConnectedAgentTool, MessageTextContent, RunStepConnectedAgentToolCall
from azure.ai.agents.models import AgentThreadCreationOptions, ThreadMessageOptions
from azure.identity import DefaultAzureCredential

project_endpoint = os.getenv("AZURE_AI_AGENT_ENDPOINT")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

# Cliente do projeto — para criar/gerir agentes (visíveis no portal!)
project = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# Cliente de agentes — para threads, runs e connected agents
agents_client = project.agents

print(f"✅ Project Client conectado: {project_endpoint}")
print(f"   Modelo: {model}")

✅ Project Client conectado: https://foundry-mod2.services.ai.azure.com/api/projects/proj-tutor
   Modelo: gpt-4o


## 🖥️ Exercício 5b.1: Pipeline Sequential de 3 Agentes

Cenário: Processar um pedido de proposta técnica com 3 agentes **server-side**:

1. **Investigador** → Analisa requisitos e identifica tecnologias
2. **Arquiteto** → Desenha a solução técnica
3. **Redator** → Cria a proposta final formatada

O **Orquestrador** usa `ConnectedAgentTool` para delegar sequencialmente.

In [ ]:
# Exercício 5b.1: Criar os 3 sub-agentes no Foundry Agent Service
from azure.ai.agents import AgentsClient

agents = AgentsClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# Agente 1: Investigador
investigador = agents.create_agent(
    model=model,
    name="investigador",
    instructions="""Es um analista tecnico. A tua tarefa e:
1. Analisar requisitos de um projeto
2. Identificar tecnologias Azure relevantes
3. Listar riscos e consideracoes
Responde em formato estruturado com seccoes claras. Portugues de Portugal.""",
)
print(f"✅ Investigador criado: {investigador.id}")

# Agente 2: Arquiteto
arquiteto = agents.create_agent(
    model=model,
    name="arquiteto",
    instructions="""Es um arquiteto de solucoes Azure. A tua tarefa e:
1. Receber uma analise tecnica
2. Desenhar uma arquitetura Azure (servicos, integracoes, fluxos)
3. Estimar dimensionamento
Usa diagramas ASCII quando possivel. Portugues de Portugal.""",
)
print(f"✅ Arquiteto criado: {arquiteto.id}")

# Agente 3: Redator
redator = agents.create_agent(
    model=model,
    name="redator",
    instructions="""Es um redator tecnico. A tua tarefa e:
1. Receber analise e arquitetura tecnica
2. Criar uma proposta profissional e concisa
3. Incluir: Resumo Executivo, Solucao Proposta, Cronograma, Proximos Passos
Maximo 400 palavras. Portugues de Portugal.""",
)
print(f"✅ Redator criado: {redator.id}")

# Criar ConnectedAgentTool para cada sub-agente
connected_tools = []
for agent in [investigador, arquiteto, redator]:
    tool = ConnectedAgentTool(
        id=agent.id,
        name=agent.name,
        description=f"Agente especializado: {agent.name}",
    )
    connected_tools.extend(tool.definitions)

# Agente Orquestrador — delega sequencialmente aos sub-agentes
orquestrador = agents.create_agent(
    model=model,
    name="orquestrador",
    instructions="""Es um gestor de projeto. Para cada pedido, executa estes passos em sequencia:
1. Primeiro, pede ao investigador para analisar os requisitos
2. Depois, pede ao arquiteto para desenhar a solucao baseada na analise do investigador
3. Por fim, pede ao redator para criar a proposta final baseada na analise e arquitetura
Apresenta o resultado final do redator ao utilizador.""",
    tools=connected_tools,
)
print(f"✅ Orquestrador criado: {orquestrador.id}")
print(f"   Connected agents: investigador, arquiteto, redator")
print(f"\n💡 Vai ao portal do Foundry e verifica que os 4 agentes aparecem la!")

✅ Agente Investigador: investigador
✅ Agente Arquiteto: arquiteto
✅ Agente Redator: redator


In [ ]:
# Exercício 5b.1: Executar o workflow sequential

pedido = """
A empresa Contoso quer migrar o seu sistema de atendimento ao cliente para a cloud.
Atualmente usam:
- Base de dados SQL Server on-premises com 500GB de dados
- Aplicacao web .NET Framework 4.8
- Chatbot basico baseado em regras

Requisitos:
- Chatbot inteligente com IA (entender perguntas em linguagem natural)
- Pesquisa nos manuais de produto (3000 documentos PDF)
- Integracao com o CRM existente (Dynamics 365)
- Alta disponibilidade (99.9% SLA)
- RGPD compliance (dados na Europa)
"""

print("=" * 60)
print("📋 PEDIDO DO CLIENTE")
print("=" * 60)
print(pedido)
print("⚙️ A executar workflow: orquestrador → investigador → arquiteto → redator...")
print("   (o Foundry Agent Service gere a comunicacao entre agentes)\n")

# Executar — create_thread_and_process_run faz polling automatico
run = agents.create_thread_and_process_run(
    agent_id=orquestrador.id,
    thread=AgentThreadCreationOptions(
        messages=[ThreadMessageOptions(
            role="user",
            content=f"Analisa os seguintes requisitos de projeto e cria uma proposta:\n{pedido}",
        )]
    ),
)
print(f"✅ Run concluido! Status: {run.status}")

# Mostrar os run steps — cada delegacao a um sub-agente aparece como tool_call
steps = list(agents.run_steps.list(thread_id=run.thread_id, run_id=run.id))
print(f"\n📋 {len(steps)} Run Steps (ordem de execucao):")
for step in reversed(steps):
    print(f"  [{step.type}] {step.status}")
    if hasattr(step.step_details, "tool_calls"):
        for tc in step.step_details.tool_calls:
            if isinstance(tc, RunStepConnectedAgentToolCall):
                print(f"    → Delegou ao agente: {tc.connected_agent.name}")

# Mostrar resposta final
if run.status == "completed":
    messages = list(agents.messages.list(thread_id=run.thread_id))
    for msg in messages:
        if msg.role == "assistant":
            for content in msg.content:
                if isinstance(content, MessageTextContent):
                    print(f"\n{'=' * 60}")
                    print("📝 PROPOSTA FINAL (gerada pelo workflow de 3 agentes)")
                    print("=" * 60)
                    print(content.text.value)
            break

print("\n✅ Pipeline sequential de 3 agentes concluida!")

✅ Workflow construído: investigador → arquiteto → redator

📋 PEDIDO DO CLIENTE

A empresa Contoso quer migrar o seu sistema de atendimento ao cliente para a cloud.
Atualmente usam:
- Base de dados SQL Server on-premises com 500GB de dados
- Aplicação web .NET Framework 4.8
- Chatbot básico baseado em regras

Requisitos:
- Chatbot inteligente com IA (entender perguntas em linguagem natural)
- Pesquisa nos manuais de produto (3000 documentos PDF)
- Integração com o CRM existente (Dynamics 365)
- Alta disponibilidade (99.9% SLA)
- RGPD compliance (dados na Europa)


⚙️ A executar pipeline de 3 agentes...

🔍 PASSO 1: Investigador
# Análise de Requisitos do Projeto

## 1. Requisitos
### Funcionais
1. **Chatbot inteligente com IA**: O sistema deve ter um chatbot com capacidades de Inteligência Artificial para compreender linguagem natural e interagir com os clientes de forma eficaz.
2. **Pesquisa nos manuais de produto**: Deve ser possível realizar pesquisas eficientes em textos contidos em 

## 🖥️ Exercício 5b.2: Pipeline de Analise de Feedback

Segundo workflow com 2 sub-agentes + orquestrador:
1. **Classificador** → Categoriza e classifica sentimento
2. **Relator** → Gera relatorio executivo

In [ ]:
# Exercício 5b.2: Pipeline de analise de feedback com 2 sub-agentes

# Sub-agente: Classificador
classificador = agents.create_agent(
    model=model,
    name="classificador",
    instructions="""Classifica cada feedback com:
- sentimento: positivo/negativo/neutro
- categoria: UX, documentacao, preco, funcionalidade
- prioridade: alta/media/baixa
Responde em JSON valido como array de objetos.""",
)
print(f"✅ Classificador: {classificador.id}")

# Sub-agente: Relator
relator = agents.create_agent(
    model=model,
    name="relator",
    instructions="""Com base na classificacao dos feedbacks, gera um relatorio executivo de 5 linhas com:
resumo geral, pontos fortes, pontos a melhorar, e uma recomendacao prioritaria.
Portugues de Portugal.""",
)
print(f"✅ Relator: {relator.id}")

# Orquestrador de feedback
connected_fb = []
for agent in [classificador, relator]:
    tool = ConnectedAgentTool(id=agent.id, name=agent.name, description=f"Agente: {agent.name}")
    connected_fb.extend(tool.definitions)

orq_feedback = agents.create_agent(
    model=model,
    name="orq_feedback",
    instructions="""Es um gestor de qualidade. Para cada conjunto de feedbacks:
1. Primeiro, pede ao classificador para categorizar cada feedback
2. Depois, pede ao relator para gerar o relatorio executivo com base na classificacao
Apresenta o relatorio final ao utilizador.""",
    tools=connected_fb,
)
print(f"✅ Orquestrador feedback: {orq_feedback.id}")

feedbacks = [
    "O portal do Foundry e muito intuitivo, adorei a experiencia de deploy dos modelos!",
    "Demorei 3 horas a configurar a autenticacao. A documentacao e confusa.",
    "O preco e razoavel para empresas, mas startups precisam de tier gratuito maior.",
    "O code interpreter nos agentes e fantastico para analise de dados.",
]

input_text = "\n".join([f"{i+1}. {f}" for i, f in enumerate(feedbacks)])

print("\n⚙️ A executar pipeline de feedback...")
run_fb = agents.create_thread_and_process_run(
    agent_id=orq_feedback.id,
    thread=AgentThreadCreationOptions(
        messages=[ThreadMessageOptions(role="user", content=f"Analisa estes feedbacks:\n{input_text}")]
    ),
)
print(f"✅ Run concluido! Status: {run_fb.status}")

# Run steps
steps_fb = list(agents.run_steps.list(thread_id=run_fb.thread_id, run_id=run_fb.id))
print(f"\n📋 {len(steps_fb)} Run Steps:")
for step in reversed(steps_fb):
    if hasattr(step.step_details, "tool_calls"):
        for tc in step.step_details.tool_calls:
            if isinstance(tc, RunStepConnectedAgentToolCall):
                print(f"  → Delegou ao agente: {tc.connected_agent.name}")

# Resposta final
if run_fb.status == "completed":
    messages_fb = list(agents.messages.list(thread_id=run_fb.thread_id))
    for msg in messages_fb:
        if msg.role == "assistant":
            for content in msg.content:
                if isinstance(content, MessageTextContent):
                    print(f"\n{'=' * 60}")
                    print("📊 RELATORIO DE FEEDBACK")
                    print("=" * 60)
                    print(content.text.value)
            break

print("\n✅ Pipeline de feedback concluida!")

In [ ]:
# Limpeza — eliminar todos os agentes criados no Foundry
for agent_id in [orquestrador.id, investigador.id, arquiteto.id, redator.id,
                 orq_feedback.id, classificador.id, relator.id]:
    try:
        agents.delete_agent(agent_id)
        print(f"🗑️ Agente eliminado: {agent_id}")
    except Exception as e:
        print(f"⚠️ Erro: {e}")

print("\n✅ Limpeza concluida! Verifica no portal do Foundry que os agentes desapareceram.")

## 🧪 Passo 2: Workflow Agent Declarativo no Portal (Preview)

### O que é um Workflow Agent?

Além da abordagem **ConnectedAgentTool** (usada acima), o Foundry v2 introduz o conceito de **Workflow Agent** — um agente declarativo cujo comportamento é definido por um ficheiro **CSDL YAML** (Conversational State Description Language).

Com esta abordagem:
- O **workflow é um recurso** dentro do Foundry (visível no portal como um "agente" do tipo Workflow)
- A lógica de orquestração é **declarativa** — não depende de instruções em linguagem natural
- O Foundry gere a execução, checkpoints e estado da conversa automaticamente

### Comparação: ConnectedAgentTool vs Workflow Agent

| Aspeto | ConnectedAgentTool (acima) | Workflow Agent (CSDL) |
|--------|---------------------------|----------------------|
| **Orquestração** | Instruções do orquestrador (LLM decide) | Declarativa (YAML define a sequência) |
| **Controlo** | Probabilístico (depende do modelo) | Determinístico (passos fixos) |
| **Criação** | Código (`create_agent` + tools) | Portal / YAML |
| **Estado** | Thread gerido pelo agente | Checkpoints automáticos |
| **Maturidade** | ✅ GA (produção) | 🧪 Preview |

---

### 📋 Passo a passo: Criar Workflow Agent no Portal do Azure AI Foundry

#### Pré-requisito
Os sub-agentes (**investigador**, **arquiteto**, **redator**) já devem existir no projeto — criámo-los no Exercício 5b.1 acima.

#### 1. Aceder ao portal
- Ir a [https://ai.azure.com](https://ai.azure.com)
- Selecionar o projeto (ex: `proj-tutor`)

#### 2. Navegar para Agents
- No menu lateral, clicar em **"Agents"**
- Verás os sub-agentes já criados (investigador, arquiteto, redator)

#### 3. Criar novo agente do tipo Workflow
- Clicar em **"+ New agent"**
- Selecionar o tipo **"Workflow"** (em vez de "Prompt")
- Dar o nome: `workflow_proposta`

#### 4. Definir o CSDL YAML
No editor de workflow, colar a seguinte definição:

```yaml
kind: Workflow
triggers:
  - kind: UserMessage
actions:
  - kind: InvokeAzureAgent
    agent: investigador
  - kind: InvokeAzureAgent
    agent: arquiteto
  - kind: InvokeAzureAgent
    agent: redator
```

**O que cada bloco faz:**
| Bloco | Descrição |
|-------|-----------|
| `kind: Workflow` | Declara que este agente é um workflow |
| `triggers: UserMessage` | O workflow arranca quando o utilizador envia uma mensagem |
| `actions: InvokeAzureAgent` | Cada ação invoca um sub-agente pela ordem definida |

#### 5. Publicar e testar
- Clicar em **"Save"** / **"Publish"**
- No painel de teste (chat), enviar o pedido:

> *"A empresa Contoso quer migrar o seu sistema de atendimento ao cliente para a cloud. Usam SQL Server 500GB, .NET Framework 4.8, chatbot básico. Precisam de chatbot IA, pesquisa em 3000 PDFs, integração Dynamics 365, 99.9% SLA, RGPD."*

- O Foundry executará automaticamente: **investigador → arquiteto → redator**
- Cada passo aparece no painel de execução com os seus outputs

#### 6. Verificar a execução
No painel lateral do chat, podes ver:
- ✅ Cada ação executada (InvokeAzureAgent)
- 📄 O output de cada sub-agente
- ⏱️ Tempo de execução por passo
- 🔄 Checkpoints automáticos

> ⚠️ **Nota:** O Workflow Agent está em **Preview** (requer feature flag `WorkflowAgents=V1Preview`). A interface no portal pode variar conforme a versão disponível no teu tenant.

## ✅ Resumo

Neste lab aprendeste:
- A criar **agentes server-side** no Foundry Agent Service com `AgentsClient.create_agent()`
- A usar **`ConnectedAgentTool`** para ligar sub-agentes a um orquestrador
- O padrão **sequential multi-agent workflow**: orquestrador delega tarefas em sequência
- A inspeccionar **run steps** para ver a delegação agente → agente
- A limpar agentes com `delete_agent()`
- A criar um **Workflow Agent declarativo** com `WorkflowAgentDefinition` e CSDL YAML (Preview)

### Abordagens de Multi-Agent Workflow no Foundry

| Abordagem | Técnica | Maturidade | Quando usar |
|-----------|---------|------------|-------------|
| **ConnectedAgentTool** | Orquestrador + sub-agentes via tools | ✅ GA | Workflows flexíveis, lógica adaptativa |
| **Workflow Agent (CSDL)** | Definição declarativa em YAML | 🧪 Preview | Workflows determinísticos, pipelines fixos |

### Padrões de Orquestração
| Padrão | Como funciona | Quando usar |
|--------|---------------|-------------|
| **Sequential** | Orquestrador chama agentes em ordem nas instruções | Pipelines lineares (este lab) |
| **Parallel** | Orquestrador chama múltiplos agentes no mesmo passo | Tarefas independentes |
| **Hierarchical** | Orquestrador delega a sub-orquestradores | Workflows complexos |

### Lab 3.1 vs Lab 5b
| | Lab 3.1 | Lab 5b |
|---|---------|--------|
| **Agentes** | 1 agente individual | 4+ agentes coordenados |
| **Coordenação** | Manual (código) | Automática (ConnectedAgentTool / CSDL) |
| **Contexto** | `previous_response_id` | Gerido pelo serviço |

**Próximo:** [Lab 6 - Knowledge / RAG](README-lab06-knowledge.md)